# GBM-Based JRN Estimation for 11 FK Joins x 3 Tasks on rel-avito

**Join Reproduction Number (JRN)** measures how much a foreign-key join improves
predictive performance in relational databases. This notebook demonstrates the
GBM-based JRN estimation pipeline applied to the **rel-avito** dataset:

- **11 FK joins** spanning extreme cardinality (fan-out 1.18 to 114K)
- **3 prediction tasks**: ad-ctr (regression/MAE), user-clicks (classification/AUROC), user-visits (classification/AUROC)
- **LightGBM probes** compare baseline (entity-only) vs. enriched (with join features) performance
- **Training-free proxies** (Pearson r, mutual information, conditional entropy) correlate with JRN
- **Probe-to-full validation** confirms lightweight probes rank joins similarly to full models

Key findings: SearchStream->AdsInfo JRN=1.34 (ad-ctr), VisitStream->UserInfo JRN=1.36 (user-visits),
Pearson proxy rho=0.926, Probe-to-full Spearman rho=0.825.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0',
         'seaborn==0.13.2', 'tabulate==0.9.0')

## Imports

In [ ]:
import json
import os
from collections import defaultdict
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from tabulate import tabulate

## Data Loading

Load pre-computed JRN measurements from the demo dataset. The data contains results
from running the full GBM-based JRN estimation pipeline on rel-avito (11 FK joins x 3 tasks).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-bc07ab-join-reproduction-number-epidemiology-in/main/experiment_iter3_gbm_based_jrn_e/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Dataset: {data['metadata']['dataset']}")
print(f"Tables: {data['metadata']['num_tables']}, FK joins: {data['metadata']['num_fk_joins']}")
print(f"Tasks tested: {data['metadata']['num_tasks_tested']}")
print(f"Total examples: {len(data['datasets'][0]['examples'])}")

## Configuration

Original experiment parameters used for GBM probes and full model validation.

In [ ]:
# ============================================================
# CONFIGURATION (from original experiment)
# ============================================================

SEEDS = [42, 123, 456]
PROBE_CONFIG = {
    "n_estimators": 200, "max_depth": 6, "learning_rate": 0.05,
    "subsample": 0.8, "colsample_bytree": 0.8, "verbose": -1,
}
FULL_CONFIG = {
    "n_estimators": 500, "max_depth": 8, "learning_rate": 0.05,
    "subsample": 0.8, "colsample_bytree": 0.8, "verbose": -1,
    "num_leaves": 127,
}
MAX_TRAIN_ROWS = 200_000

# Task configuration (excluding user-ad-visit link_prediction)
TASK_CONFIG = {
    "rel-avito/ad-ctr": {
        "entity_table": "AdsInfo", "entity_col": "AdID",
        "task_type": "regression", "metric": "MAE",
        "higher_is_better": False,
    },
    "rel-avito/user-clicks": {
        "entity_table": "UserInfo", "entity_col": "UserID",
        "task_type": "binary_classification", "metric": "AUROC",
        "higher_is_better": True,
    },
    "rel-avito/user-visits": {
        "entity_table": "UserInfo", "entity_col": "UserID",
        "task_type": "binary_classification", "metric": "AUROC",
        "higher_is_better": True,
    },
}

AGG_TYPES = ["mean", "sum", "max", "std", "all_combined"]

# Number of top/bottom joins to display in detailed views
N_DISPLAY = 5

## Parse JRN Measurements

Extract individual JRN measurement examples from the loaded data. Each measurement
represents one (task, FK join) pair with baseline score, enriched score, and computed JRN.

In [ ]:
# ============================================================
# PARSE JRN MEASUREMENTS
# ============================================================
examples = data["datasets"][0]["examples"]

jrn_measurements = []
proxy_correlations = None
probe_to_full = None
jrn_matrix_data = None
extreme_cardinality = None
agg_sensitivity_data = None
cross_dataset = None

for ex in examples:
    mtype = ex.get("metadata_measurement_type", "")

    if mtype == "jrn_measurement":
        inp = json.loads(ex["input"])
        out = json.loads(ex["output"])
        baseline = json.loads(ex["predict_baseline"])
        method = json.loads(ex["predict_our_method"])
        jrn_measurements.append({
            "task": inp["task_name"],
            "join_key": inp["join_key"],
            "join_idx": inp["join_idx"],
            "direction": inp["direction"],
            "fanout_mean": inp["fanout_mean"],
            "jrn": out["jrn"],
            "best_agg_type": out["best_agg_type"],
            "agg_sensitivity": out["agg_sensitivity"],
            "agg_scores": out["agg_scores"],
            "baseline_score": baseline["score"],
            "baseline_std": baseline["std"],
            "metric": baseline["metric"],
            "method_score": method["score"],
        })

    elif mtype == "proxy_correlations":
        proxy_correlations = json.loads(ex["output"])

    elif mtype == "probe_to_full":
        probe_to_full = json.loads(ex["output"])

    elif mtype == "jrn_matrix":
        jrn_matrix_data = json.loads(ex["output"])

    elif mtype == "extreme_cardinality":
        extreme_cardinality = json.loads(ex["output"])

    elif mtype == "aggregation_sensitivity":
        agg_sensitivity_data = json.loads(ex["output"])

    elif mtype == "cross_dataset":
        cross_dataset = json.loads(ex["output"])

print(f"Parsed {len(jrn_measurements)} JRN measurements")
print(f"Proxy correlations: {proxy_correlations is not None}")
print(f"Probe-to-full validation: {probe_to_full is not None}")
print(f"JRN matrix: {jrn_matrix_data is not None}")

## JRN Results Summary

Display all JRN measurements as a table. JRN > 1.0 indicates the join improves
prediction; JRN = 1.0 means no improvement (often for non-connected joins).

In [ ]:
# ============================================================
# JRN RESULTS TABLE
# ============================================================
jrn_df = pd.DataFrame(jrn_measurements)

# Display connected joins (JRN != 1.0 or direction != "none")
connected = jrn_df[jrn_df["direction"] != "none"].copy()
connected = connected.sort_values("jrn", ascending=False)

print("=" * 80)
print("JRN MEASUREMENTS — CONNECTED JOINS (direction != none)")
print("=" * 80)
table_data = []
for _, row in connected.iterrows():
    table_data.append([
        row["task"].replace("rel-avito/", ""),
        row["join_key"],
        row["direction"],
        f"{row['fanout_mean']:.1f}",
        f"{row['jrn']:.4f}",
        row["best_agg_type"],
        f"{row['baseline_score']:.6f}",
        f"{row['method_score']:.6f}",
        row["metric"],
    ])

print(tabulate(table_data,
    headers=["Task", "Join", "Dir", "Fanout", "JRN", "BestAgg", "Baseline", "Method", "Metric"],
    tablefmt="grid"))

print(f"\nTotal measurements: {len(jrn_df)}")
print(f"Connected joins: {len(connected)}")
print(f"Non-connected (JRN=1.0): {len(jrn_df) - len(connected)}")

## JRN Matrix Reconstruction

Reconstruct the JRN matrix (joins x tasks) from the parsed measurements.
This mirrors the `assemble_output` function from the original pipeline.

In [ ]:
# ============================================================
# JRN MATRIX RECONSTRUCTION
# ============================================================
tasks = sorted(TASK_CONFIG.keys())
joins = jrn_matrix_data["joins"]

# Build matrix from pre-computed data
matrix = np.array(jrn_matrix_data["jrn_matrix"])  # shape: (n_joins, n_tasks)
matrix_df = pd.DataFrame(matrix, index=joins, columns=[t.replace("rel-avito/", "") for t in tasks])

print("JRN Matrix (rows=joins, columns=tasks):")
print(matrix_df.to_string(float_format=lambda x: f"{x:.4f}"))

print(f"\nConnected joins per task:")
for task, count in jrn_matrix_data["connected_joins_per_task"].items():
    print(f"  {task}: {count} connected joins")

## Training-Free Proxy Correlations

The original pipeline computes Spearman correlations between training-free statistical
proxies (Pearson r, mutual information, conditional entropy reduction) and GBM-based JRN.
High correlations suggest these cheap proxies can approximate expensive model-based JRN.

In [ ]:
# ============================================================
# TRAINING-FREE PROXY CORRELATIONS
# ============================================================
print("Spearman correlations between training-free proxies and JRN:")
print("=" * 60)

proxy_table = []
for proxy_name, result in proxy_correlations.items():
    rho = result["rho"]
    pval = result["pval"]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    proxy_table.append([
        proxy_name.replace("_vs_jrn", ""),
        f"{rho:.4f}",
        f"{pval:.6f}",
        sig,
    ])

print(tabulate(proxy_table,
    headers=["Proxy", "Spearman rho", "p-value", "Significance"],
    tablefmt="grid"))

print("\nKey finding: Pearson r shows strongest correlation with JRN (rho=0.926),")
print("suggesting simple feature-label correlations can cheaply approximate JRN.")

## Probe-to-Full Validation

Validates that lightweight probe models (200 trees, depth 6) rank joins similarly
to full models (500 trees, depth 8). High Spearman rho confirms probes are reliable
for ranking joins by their JRN.

In [ ]:
# ============================================================
# PROBE-TO-FULL VALIDATION
# ============================================================
print("Probe-to-Full Model JRN Validation:")
print("=" * 60)
print(f"Spearman rho: {probe_to_full['spearman_rho']:.4f}")
print(f"Spearman p-value: {probe_to_full['spearman_pval']:.4f}")
print(f"Number of (task, join) pairs tested: {probe_to_full['n_pairs']}")

print("\nPairwise comparison:")
ptf_table = []
for i, (probe_jrn, full_jrn) in enumerate(zip(
    probe_to_full["probe_jrns"], probe_to_full["full_jrns"]
)):
    ptf_table.append([i + 1, f"{probe_jrn:.4f}", f"{full_jrn:.4f}",
                       f"{abs(probe_jrn - full_jrn):.4f}"])

print(tabulate(ptf_table,
    headers=["Pair", "Probe JRN", "Full JRN", "|Diff|"],
    tablefmt="grid"))

## Aggregation Sensitivity Analysis

For reverse joins (one-to-many), different aggregation strategies (mean, sum, max, std)
produce different JRN values. The sensitivity (coefficient of variation) measures how
much JRN depends on the choice of aggregation.

In [ ]:
# ============================================================
# AGGREGATION SENSITIVITY ANALYSIS
# ============================================================
if agg_sensitivity_data and "data" in agg_sensitivity_data:
    agg_items = agg_sensitivity_data["data"]
    # Filter to joins with actual variation
    varied = [a for a in agg_items if a["agg_sensitivity"] > 0.001]
    print(f"Joins with aggregation sensitivity > 0.001: {len(varied)}/{len(agg_items)}")
    print()

    agg_table = []
    for item in sorted(varied, key=lambda x: x["agg_sensitivity"], reverse=True):
        scores_str = ", ".join(f"{k}={v:.4f}" for k, v in item["agg_scores"].items())
        agg_table.append([
            item["task"].replace("rel-avito/", ""),
            item["join_key"],
            f"{item['jrn']:.4f}",
            f"{item['agg_sensitivity']:.4f}",
            scores_str,
        ])

    print(tabulate(agg_table,
        headers=["Task", "Join", "JRN", "Sensitivity (CoV)", "Agg Scores"],
        tablefmt="grid"))

## Visualization

Four-panel figure summarizing the JRN estimation results:
1. **JRN Heatmap**: Matrix of JRN values across joins and tasks
2. **JRN by Join Direction**: Comparing forward, reverse, and non-connected joins
3. **Probe vs. Full JRN**: Scatter plot validating probe model reliability
4. **Proxy Correlations**: Bar chart of training-free proxy Spearman rho values

In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("GBM-Based JRN Estimation: rel-avito (11 FK Joins x 3 Tasks)", fontsize=14, fontweight="bold")

# --- Panel 1: JRN Heatmap ---
ax1 = axes[0, 0]
# Shorten join names for display
short_joins = [j.split("->")[0].split(".")[-1] + "->" + j.split("->")[1] for j in joins]
heatmap_data = matrix_df.copy()
heatmap_data.index = short_joins

sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlOrRd",
            vmin=0.99, vmax=1.4, ax=ax1, cbar_kws={"label": "JRN"})
ax1.set_title("JRN Matrix (Joins x Tasks)")
ax1.set_ylabel("FK Join")
ax1.set_xlabel("")
ax1.tick_params(axis="x", rotation=15)
ax1.tick_params(axis="y", rotation=0, labelsize=8)

# --- Panel 2: JRN by Direction (box plot) ---
ax2 = axes[0, 1]
for direction, color, offset in [("forward", "#4CAF50", -0.2), ("reverse", "#F44336", 0.0), ("none", "#9E9E9E", 0.2)]:
    subset = jrn_df[jrn_df["direction"] == direction]
    if len(subset) > 0:
        jrn_vals = subset["jrn"].values
        ax2.scatter([offset] * len(jrn_vals) + np.random.uniform(-0.05, 0.05, len(jrn_vals)),
                    jrn_vals, alpha=0.7, label=f"{direction} (n={len(subset)})", color=color, s=60)
        ax2.hlines(np.mean(jrn_vals), offset - 0.08, offset + 0.08, color=color, linewidth=2)

ax2.axhline(y=1.0, color="black", linestyle="--", alpha=0.5, label="JRN=1.0 (no gain)")
ax2.set_ylabel("JRN")
ax2.set_title("JRN by Join Direction")
ax2.set_xticks([-0.2, 0.0, 0.2])
ax2.set_xticklabels(["Forward", "Reverse", "None"])
ax2.legend(fontsize=8)
ax2.set_ylim(0.95, 1.45)

# --- Panel 3: Probe vs Full JRN ---
ax3 = axes[1, 0]
probe_jrns = probe_to_full["probe_jrns"]
full_jrns = probe_to_full["full_jrns"]
ax3.scatter(probe_jrns, full_jrns, s=80, c="#2196F3", edgecolors="black", zorder=5)

# Add identity line
lims = [min(min(probe_jrns), min(full_jrns)) - 0.05,
        max(max(probe_jrns), max(full_jrns)) + 0.05]
ax3.plot(lims, lims, "k--", alpha=0.5, label="y=x (perfect agreement)")
ax3.set_xlabel("Probe JRN (200 trees, depth 6)")
ax3.set_ylabel("Full JRN (500 trees, depth 8)")
ax3.set_title(f"Probe vs. Full JRN (Spearman rho={probe_to_full['spearman_rho']:.3f})")
ax3.legend(fontsize=8)
ax3.set_xlim(lims)
ax3.set_ylim(lims)

# --- Panel 4: Proxy Correlations ---
ax4 = axes[1, 1]
proxy_names = []
proxy_rhos = []
proxy_colors = []
for name, result in proxy_correlations.items():
    proxy_names.append(name.replace("_vs_jrn", "").replace("_", " ").title())
    proxy_rhos.append(result["rho"])
    pval = result["pval"]
    proxy_colors.append("#4CAF50" if pval < 0.05 else "#FF9800" if pval < 0.1 else "#9E9E9E")

bars = ax4.barh(proxy_names, proxy_rhos, color=proxy_colors, edgecolor="black")
ax4.set_xlabel("Spearman rho with JRN")
ax4.set_title("Training-Free Proxy Correlations with JRN")
ax4.axvline(x=0, color="black", linewidth=0.5)

# Add value labels
for bar, rho in zip(bars, proxy_rhos):
    ax4.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
             f"{rho:.3f}", va="center", fontsize=10)

plt.tight_layout()
plt.savefig("jrn_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to jrn_results.png")

## Summary Statistics

In [ ]:
# ============================================================
# SUMMARY STATISTICS
# ============================================================
print("=" * 70)
print("EXPERIMENT SUMMARY: GBM-Based JRN Estimation on rel-avito")
print("=" * 70)

# Highest JRN values
top_jrn = connected.nlargest(N_DISPLAY, "jrn")
print(f"\nTop {N_DISPLAY} highest JRN values (most beneficial joins):")
for _, row in top_jrn.iterrows():
    print(f"  {row['join_key']:45s} | {row['task'].replace('rel-avito/', ''):15s} | JRN={row['jrn']:.4f}")

# Extreme cardinality range
fanouts = jrn_df["fanout_mean"]
print(f"\nCardinality range: {fanouts.min():.2f} to {fanouts.max():.0f} (fan-out)")

# Proxy summary
print(f"\nTraining-free proxy correlations with JRN:")
for name, result in proxy_correlations.items():
    print(f"  {name.replace('_vs_jrn', ''):25s} rho={result['rho']:.4f} (p={result['pval']:.6f})")

print(f"\nProbe-to-full Spearman rho: {probe_to_full['spearman_rho']:.3f} "
      f"(p={probe_to_full['spearman_pval']:.4f})")

# Cross-dataset note
if cross_dataset:
    print(f"\nCross-dataset note: {cross_dataset.get('note', 'N/A')}")

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)